In [ ]:
import os
import subprocess
import pandas as pd
import tifffile
from matplotlib import pyplot as plt
from PIL import Image
import sys
import cv2
import numpy as np

In [ ]:
dataset = "cos_hsa_prostate_c"

In [ ]:
data_dir = f"PATH/TO/{dataset}/"
with open(os.path.join(data_dir, "pixel-size-raw.txt"), encoding="utf-8") as handle:
    pixel_size = float(handle.read())
if not np.isfinite(pixel_size) or pixel_size <= 0:
    raise ValueError("The aligned H&E pixel size must be positive")


In [ ]:
with open(f"{data_dir}pixel-size-raw.txt","w") as f:
    f.write(str(pixel_size))

In [ ]:
with open(f"{data_dir}radius.txt","w") as f:
    f.write(str(round(55 / 2 / pixel_size)))

In [ ]:
with open(f"{data_dir}radius-raw.txt","w") as f:
    f.write(str(55 / 2 / pixel_size))

In [ ]:
he = tifffile.imread(f"{data_dir}he-raw.tiff")
print(he.shape)
plt.imshow(he)

In [ ]:
he_pil = Image.fromarray(he)
he_pil.save(f"{data_dir}he-raw.jpg")

In [ ]:
num_px_height = round(he.shape[0] * pixel_size / 8)
num_px_width = round(he.shape[1] * pixel_size / 8)
target_s = round(64*(max(num_px_height, num_px_width)//64+1))
blank_height = round(8 / pixel_size * (target_s - num_px_height))
blank_width = round(8 / pixel_size * (target_s - num_px_width))
print("target_s:\t" + str(target_s))
print("num_px_height:\t" + str(num_px_height))
print("num_px_width:\t" + str(num_px_width))
print("blank_height:\t" + str(blank_height))
print("blank_width:\t" + str(blank_width))

In [ ]:
width = he.shape[1]
height = blank_height + 1
img = np.zeros((height, width, 3), np.uint8)
img_rgb = img.copy()
img_rgb[:, :, :] = [255, 255, 255]
he_sq = np.concatenate((he,img_rgb))

width = blank_width
height = he.shape[0] + blank_height + 1
img = np.zeros((height, width, 3), np.uint8)
img_rgb = img.copy()
img_rgb[:, :, :] = [255, 255, 255]
he_sq = np.concatenate((he_sq,img_rgb), axis=1)

In [ ]:
he_sq.shape

In [ ]:
plt.imshow(he_sq)

In [ ]:
superst_image_dir = f"{data_dir}/superST_data/"
os.makedirs(superst_image_dir, exist_ok = True)
tifffile.imwrite(f'{superst_image_dir}tissue_hires_image.tiff', he_sq)

In [ ]:
he_sq_pil = Image.fromarray(he_sq)
he_sq_pil.save(
    f'{superst_image_dir}tissue_hires_image.png', 
    format='PNG'
)
he_sq_pil.save(
    f'{superst_image_dir}tissue_lowres_image.png', 
    format='PNG'
)